### Pitch Control analysis in the WC2022 using pff data

In [ ]:
# Import libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from kloppy import pff
from databallpy import get_game_from_kloppy
from Utils.config import BASE_PATH
from Utils.loading import load_files, load_game_from_pff
from matplotlib.colors import LinearSegmentedColormap
from databallpy.visualize import plot_soccer_pitch, plot_tracking_data

In [ ]:
# Load event and tracking data files
dict_events, dict_tracking = load_files(BASE_PATH)

In [ ]:
for game_id in dict_events.keys():
    events_df = dict_events[game_id]
    tracking_df = dict_tracking[game_id]

# Create a game object
game = load_game_from_pff(BASE_PATH, game_id)

In [ ]:
# Get a shot in events_df
shot_event = events_df[events_df['possessionEvents.possessionEventType'] == 'SH'].iloc[0]
# Find the corresponding shot in tracking_df
shot_tracking = tracking_df[tracking_df["possession_event_id"] == shot_event["possessionEventId"]].iloc[0]

def get_frame(tracking_passes_df):
    tracking_passes_df["possession_start_frame"] = tracking_passes_df["possession_event"].apply(
        lambda x: x.get("start_frame") if isinstance(x, dict) else None)
    return tracking_passes_df

shot_tracking = get_frame(shot_tracking)
# Get the start frame of the possession event for this pass
possession_start_frame = shot_tracking.get("possession_start_frame")
game_tracking_frame = game.tracking_data[game.tracking_data["frame"] == possession_start_frame].iloc[0]

idx_frame = game_tracking_frame.index[0]
print(idx_frame)
pitch_control = game.tracking_data.get_pitch_control(game.pitch_dimensions, 105, 68, idx_frame,idx_frame)
print(pitch_control.shape)
cmap_red_green = LinearSegmentedColormap.from_list("reds", [(0, 1, 0, 1), (0.5, 0.5, 0, 0), (1, 0, 0, 1)])

fig, ax = plot_soccer_pitch(field_dimen=game.pitch_dimensions, pitch_color="white")
fig, ax = plot_tracking_data(game, idx_frame, team_colors=["green", "red"], ax=ax, fig=fig, add_velocities=True, heatmap_overlay=pitch_control[0], overlay_cmap=cmap_red_green)

plt.show()